In [5]:
import pandas as pd
import requests
import time
from tqdm import tqdm

# =============================
# CONFIG
# =============================
WEATHER_API_KEY = "49f8c6e8ff3b4c938ba95239260604"
INPUT_FILE = "nashik_onion_clean.csv"
OUTPUT_FILE = "nashik_onion_final.csv"

# =============================
# LOAD DATA
# =============================
df = pd.read_csv(INPUT_FILE)
df = df.rename(columns={
    "market_name": "Market",
    "variety": "Grade",
    "p_min": "Min Price (Rs./ Qtl)",
    "p_max": "Max Price (Rs./Qtl)",
    "p_modal": "Modal Price (Rs./Qtl)",
    "t": "Date"
})

df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

df["Market"] = df["Market"].str.upper().str.strip()
df["Grade"] = df["Grade"].str.upper().str.strip()

# =============================
# TIME FEATURES
# =============================
df["Day"] = df["Date"].dt.day
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

# =============================
# COORDINATES
# =============================
coords = {
    "NASIK": (19.9975, 73.7898),
    "LASALGAON": (20.1420, 74.2390),
    "PIMPALGAON": (20.2000, 74.0000),
    "CHANDVAD": (20.3300, 74.2500),
    "MANMAD": (20.2500, 74.4500),
    "SINNER": (19.8500, 74.0000),
    "DEVALA": (20.4000, 74.5000),
    "LASALGAON(NIPHAD)": (20.1420, 74.2390),
    "PIMPALGAON BASWANT(SAYKHEDA)": (20.1300, 73.9000),
    "LASALGAON(VINCHUR)": (20.1500, 74.2000),
    "NAMPUR": (20.4700, 74.2000),
    "SHIVSIDDHA GOVIND PRODUCER COMPANY LIMITED SANCHAL": (20.0000, 74.0000),
    "YEOLA": (20.0500, 74.5000),
    "UMRANE": (20.3000, 74.2000),
    "SATANA": (20.6000, 74.0000),
    "KALVAN": (20.5000, 74.0000),
    "DINDORI(VANI)": (20.2000, 73.9000),
    "NANDGAON": (20.3000, 74.6500),
    "MALHARSHREE FARMERS PRODUCER CO LTD": (20.1000, 74.2000),
    "DINDORI": (20.2000, 73.9000),
    "SHREE RAMESHWAR KRUSHI MARKET": (20.1000, 74.1000),
    "MANKAMNESHWAR FARMAR PRODUCER COLTD SANCHALIT MANK": (20.1500, 74.1500),
    "SURAGANA": (20.5500, 73.6500)
}

# =============================
# FESTIVAL FUNCTION
# =============================
def get_festival(date):
    try:
        url = f"https://calender-api-production.up.railway.app/calendar?date={date_str}"
        res = requests.get(url).json()

        fest_list = res.get("state_festivals", {}).get("Maharashtra", [])
        festival = ", ".join(fest_list) if fest_list else "None"

        return (
            festival,
            int(res.get("is_amavasya", False)),
            int(res.get("is_ekadashi", False)),
            int(res.get("is_purnima", False)),
            res.get("tithi", "Unknown"),
            res.get("nakshatra", "Unknown")
        )
    except:
        return "None", 0, 0, 0, "Unknown", "Unknown"

# =============================
# WEATHER FUNCTION
# =============================
def get_weather(lat, lon, date):
    try:
        url = "http://api.weatherapi.com/v1/history.json"

        params = {
            "key": WEATHER_API_KEY,
            "q": f"{lat},{lon}",
            "dt": date
        }

        res = requests.get(url, params=params).json()
        day = res["forecast"]["forecastday"][0]["day"]

        temp = day["maxtemp_c"]
        rain = day["totalprecip_mm"]

    except:
        temp, rain = 30, 0

    heat = "Yes" if temp > 35 else "No"

    if rain > 20:
        rain_alert = "Heavy"
    elif rain > 5:
        rain_alert = "Moderate"
    else:
        rain_alert = "No"

    return temp, rain, heat, rain_alert

# =============================
# AGRO ADVISORY
# =============================
def get_agro_advisory(temp, rain, heat, rain_alert):

    if heat == "Yes" and rain < 2:
        return "Irrigation needed | Heat stress risk"
    elif rain_alert == "Heavy":
        return "Heavy rain expected"
    elif rain > 5:
        return "Good moisture"
    else:
        return "Normal conditions"

# =============================
# MAIN LOOP WITH TQDM
# =============================
results = []

for i, row in tqdm(df.iterrows(), total=len(df)):

    date = row["Date"].strftime("%Y-%m-%d")
    market = row["Market"]

    lat, lon = coords.get(market, (19.9975, 73.7898))

    # FESTIVAL
    festival, ama, eka, pur, tithi, nak = get_festival(date)

    # WEATHER
    temp, rain, heat, rain_alert = get_weather(lat, lon, date)

    # ADVISORY
    advisory = get_agro_advisory(temp, rain, heat, rain_alert)

    results.append({
        "Date": date,
        "Market": market,
        "Grade": row["Grade"],

        "Min Price (Rs./ Qtl)": row["Min Price (Rs./ Qtl)"],
        "Max Price (Rs./Qtl)": row["Max Price (Rs./Qtl)"],
        "Modal Price (Rs./Qtl)": row["Modal Price (Rs./Qtl)"],

        "Festival": festival,
        "is_amavasya": ama,
        "is_ekadashi": eka,
        "is_purnima": pur,
        "Tithi": tithi,
        "Nakshatra": nak,

        "Day": row["Day"],
        "Month": row["Month"],
        "Year": row["Year"],
        "Week": row["Week"],

        "Temp Max (°C)": temp,
        "Rain (mm)": rain,
        "Heat Stress": heat,
        "Rain Alert": rain_alert,

        "Agro Advisory": advisory
    })

    time.sleep(0.2)  # avoid API block

# =============================
# SAVE
# =============================
final_df = pd.DataFrame(results)
final_df.to_csv(OUTPUT_FILE, index=False)

print("🔥 DONE! FILE SAVED:", OUTPUT_FILE)

100%|██████████| 16337/16337 [2:32:48<00:00,  1.78it/s]  


🔥 DONE! FILE SAVED: nashik_onion_final.csv
